# 06 — S&OP End-to-End Model
This notebook builds the Day 6 S&OP engine: Demand, Supply, Inventory Rollforward, and Scenario Adjustments.

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np

from src.modules.scenario_adjustments import apply_scenario_multiplier
from src.modules.demand import build_demand_module
from src.modules.supply import build_supply_module
from src.modules.inventory import build_inventory_module

## 1. Base Demand & Seasonality

In [ ]:
time = pd.period_range('2026Q1', periods=8, freq='Q')
products = ['Core','Premium']

base_demand = xr.DataArray(
    np.tile([[5000, 1200]], (8,1)),
    dims=('time','product'),
    coords={'time':time,'product':products}
)

seasonality = xr.DataArray(
    [1.00,1.05,1.10,1.00, 0.95,1.02,1.08,1.00],
    dims=['time'], coords={'time':time}
)
base_demand, seasonality

## 2. Apply Scenario Multipliers

In [ ]:
scenarios = ['Base','Upside','Downside']
multipliers = [1.00, 1.07, 0.93]

adj_demand = apply_scenario_multiplier(base_demand, scenarios, multipliers)
adj_demand

## 3. Build Full Demand Module

In [ ]:
demand_ds = build_demand_module(base_demand, seasonality, adj_demand)
demand_ds

## 4. Build Supply Module (capacity limits)

In [ ]:
capacity = {'Base':7000, 'Upside':7500, 'Downside':6500}
production_ratio = 1.0  # aim to meet demand

supply = build_supply_module(capacity, production_ratio, scenarios, time, products)
supply

## 5. Inventory Rollforward

In [ ]:
opening_inventory = 3000
inv_ds = build_inventory_module(demand_ds['SeasonalDemand'], supply, opening_inventory)
inv_ds